# SchemaQuake GRPO Training

End-to-end LoRA fine-tuning of a small instruct model on the SchemaQuake environment using [Unsloth](https://github.com/unslothai/unsloth) for 4-bit loading and [TRL's GRPO trainer](https://huggingface.co/docs/trl) for policy optimization.

**Target:** Qwen2.5-3B-Instruct on a single A100 40GB.  
**Time:** ~6–10 hours for a visible reward curve.

## 1. Install

In [ ]:
!pip -q install unsloth 'trl>=0.11' 'transformers>=4.44' 'peft>=0.12' 'accelerate>=0.33' 'datasets>=2.20' 'bitsandbytes>=0.43' wandb openenv-core>=0.2.3
!pip -q install -e /content/schemaquake  # or: pip install git+https://github.com/<you>/schemaquake

## 2. Load model with Unsloth

In [5]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
    use_gradient_checkpointing='unsloth',
)

ModuleNotFoundError: No module named 'unsloth'

## 3. Episode rollout + reward extraction

We wrap the SchemaQuake environment so a full episode becomes a single (prompt, completion, scalar reward) tuple, which is exactly what GRPO consumes.

In [ ]:
import json, re
from schemaquake import SchemaQuakeEnv, SQAction, ToolName
from schemaquake.prompts import SYSTEM_PROMPT

MAX_STEPS = 20

_TOOL_JSON_RE = re.compile(r'\{[^{}]*?\}', re.DOTALL)

def parse_action(text: str) -> SQAction:
    m = _TOOL_JSON_RE.search(text)
    if not m:
        return SQAction(tool=ToolName.NOOP, confidence=0.0)
    try:
        d = json.loads(m.group(0))
        return SQAction(
            tool=ToolName(d.get('tool','noop')),
            args=d.get('args') or {},
            confidence=d.get('confidence'),
        )
    except Exception:
        return SQAction(tool=ToolName.NOOP, confidence=0.0)

def build_prompt(history):
    msgs = [{'role':'system','content':SYSTEM_PROMPT}]
    for turn in history:
        msgs.append(turn)
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def rollout_one(seed:int):
    env = SchemaQuakeEnv(max_steps=MAX_STEPS, p_no_drift=0.2)
    obs = env.reset(seed=seed, episode_id=f'tr-{seed}')
    history = [{'role':'user','content':json.dumps(obs.episode_brief)}]
    generations = []
    while not obs.done:
        prompt = build_prompt(history)
        ids = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**ids, max_new_tokens=96, do_sample=True, temperature=0.7, top_p=0.9)
        text = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)
        generations.append(text)
        act = parse_action(text)
        obs = env.step(act)
        history.append({'role':'assistant','content':text})
        history.append({'role':'user','content':json.dumps(obs.tool_result)[:1200]})
    total = (obs.reward_breakdown or {}).get('total', 0.0)
    return build_prompt(history[:1]), '\n'.join(generations), float(total), obs.reward_breakdown

## 4. GRPO trainer

We use TRL's `GRPOTrainer` with a custom `reward_funcs` that re-runs the episode and returns the total SchemaQuake reward. Because our environment reward is already a dense, bounded scalar, GRPO converges cleanly without a separate value model.

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# A tiny synthetic dataset — one row per seed. The actual 'prompt' is just the
# episode seed; reward_fn ignores the prompt and runs a fresh episode on a seed.
NUM_PROMPTS = 512
train_ds = Dataset.from_list([{'prompt': f'seed:{i}', 'seed': i} for i in range(NUM_PROMPTS)])

def schemaquake_reward(completions, prompts=None, **kw):
    rewards = []
    for p in prompts:
        seed = int(p.split(':')[-1])
        _, _, total, _ = rollout_one(seed)
        rewards.append(total)
    return rewards

cfg = GRPOConfig(
    output_dir='runs/schemaquake_grpo',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    num_generations=8,
    max_prompt_length=512,
    max_completion_length=192,
    logging_steps=5,
    save_steps=50,
    report_to='wandb',
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=schemaquake_reward,
    args=cfg,
    train_dataset=train_ds,
    processing_class=tokenizer,
)
trainer.train()
trainer.save_model('runs/schemaquake_grpo/final')

## 5. Evaluate before vs after

Export the adapter, run `python -m eval.run_eval --agent llm --episodes 50` with your base model endpoint then with your trained endpoint, and compare the 4-panel plots.